In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install spectral
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.0/249.0 kB 6.3 MB/s eta 0:00:00


In [8]:
# Full end-to-end script: training, evaluation, saving maps & reports, zipping outputs.
# Option B: include border pixels by padding; exact samples_per_class training.
import os, time, zipfile, gc
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, accuracy_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
import spectral

# ---------------- Config ----------------
dataset = 'IP'   # 'IP','SA','PU','Ho','Bo','KSC'
base = "/content/drive/MyDrive/Colab Notebooks/dataset/"   # <-- change to your path
windowSize = 25
#samples_per_class = 15
train_ratio = 0.1   # <-- 10% per class
batch_size = 256
epochs = 100
results_folder = f"ulite_results_{dataset}_spc{samples_per_class}"
os.makedirs(results_folder, exist_ok=True)

# Allow mixed precision if desired (optional)
try:
    mixed_precision.set_global_policy("mixed_float16")
except Exception:
    pass
tf.keras.backend.clear_session(); gc.collect()

K_default = 30 if dataset == 'IP' else 15
K = K_default

# ---------------- Data loaders ----------------
def loadData(name):
    if name == 'IP':
        data = sio.loadmat(os.path.join(base, 'Indian_pines_corrected.mat'))['indian_pines_corrected']
        labels = sio.loadmat(os.path.join(base, 'Indian_pines_gt.mat'))['indian_pines_gt']
    elif name == 'SA':
        data = sio.loadmat(os.path.join(base, 'Salinas_corrected.mat'))['salinas_corrected']
        labels = sio.loadmat(os.path.join(base, 'Salinas_gt.mat'))['salinas_gt']
    elif name == 'Ho':
        data = sio.loadmat(os.path.join(base, 'Houston.mat'))['houston']
        labels = sio.loadmat(os.path.join(base, 'Houston_gt.mat'))['houston_gt']
    elif name == 'PU':
        data = sio.loadmat(os.path.join(base, 'PaviaU.mat'))['paviaU']
        labels = sio.loadmat(os.path.join(base, 'PaviaU_gt.mat'))['paviaU_gt']
    elif name == 'Bo':
        data = sio.loadmat(os.path.join(base, 'Botswana.mat'))['Botswana']
        labels = sio.loadmat(os.path.join(base, 'Botswana_gt.mat'))['Botswana_gt']
    elif name == 'KSC':
        data = sio.loadmat(os.path.join(base, 'KSC.mat'))['KSC']
        labels = sio.loadmat(os.path.join(base, 'KSC_gt.mat'))['KSC_gt']
    else:
        raise ValueError("Dataset not supported")
    return data, labels

def applyPCA(X, numComponents):
    Xr = X.reshape(-1, X.shape[2]).astype(np.float32)
    pca = PCA(n_components=numComponents, whiten=True)
    Xp = pca.fit_transform(Xr)
    return Xp.reshape(X.shape[0], X.shape[1], numComponents), pca

def padWithZeros(X, margin=0):
    return np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')

# ---------------- Patch generator ----------------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        """
        coords: array of (r,c) in ORIGINAL image coordinates (0..H-1 / 0..W-1)
        The generator pads full_cube internally by half=patch_size//2 and extracts:
            start = r,   slice padded[start : start+patch_size]
        which works because padded has top-left padding of 'half' rows/cols.
        """
        self.coords = np.array(coords, dtype=np.int32)
        self.labels = np.array(labels, dtype=np.int32)
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(self.labels))
        # pad once (so padded index i corresponds to original index i-half)
        self.padded = padWithZeros(full_cube, self.half)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            # In padded array the top-left of a patch centered at (r,c) original is at index r
            # because padded has half rows/cols at top/left.
            r0 = r
            c0 = c
            patch = self.padded[r0:r0+self.patch_size, c0:c0+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y

# ---------------- Fixed Split function (samples_per_class) ----------------
'''def splitTrainTestSet(coords, labels, samples_per_class, random_state=42):
    """
    coords, labels are arrays with same length. labels in [0..n-1]
    Returns coords_train, coords_test, labels_train, labels_test
    Training picks up to samples_per_class from each class (if available).
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = min(samples_per_class, len(idx))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]'''


    ######### Train Ratio fixed ###################
def splitTrainTestSet_ratio(coords, labels, train_ratio, random_state=42):
    """
    coords, labels arrays with same length.
    Takes train_ratio fraction from each class for training.
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = max(1, int(len(idx) * train_ratio))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]


# ---------------- ULite-R2HCN blocks (simplified/robust) ----------------
from tensorflow.keras import layers

def R2SpectralBlock(x):
    in_ch = int(x.shape[-1])
    y = layers.Conv3D(filters=max(8, in_ch), kernel_size=(1,1,1), padding='same', activation='relu')(x)
    y = layers.BatchNormalization()(y)
    y = layers.Conv3D(filters=max(8, in_ch//2 + 1), kernel_size=(1,1,1), padding='same', activation='relu')(y)
    y = layers.BatchNormalization()(y)
    return y

def R2SpatialBlock(x):
    y = layers.Conv3D(filters=max(16, int(x.shape[-1])), kernel_size=(3,3,3), padding='same', activation='relu')(x)
    y = layers.BatchNormalization()(y)
    # collapse spectral & channel dims
    h = y.shape[1]; w = y.shape[2]; d = y.shape[3]; ch = y.shape[4]
    y_resh = layers.Reshape((h, w, d*ch))(y)
    y_resh = layers.SeparableConv2D(filters=max(16, ch*2), kernel_size=(3,3), padding='same', activation='relu')(y_resh)
    y_resh = layers.BatchNormalization()(y_resh)
    y_out = layers.Reshape((h, w, 1, int(y_resh.shape[-1])))(y_resh)
    return y_out

def build_ulite_r2hcn(windowSize, K, num_classes):
    inp = layers.Input(shape=(windowSize, windowSize, K, 1), dtype='float32')
    x = layers.Conv3D(filters=8, kernel_size=(3,3,7), padding='valid', activation='relu')(inp)
    x = layers.Conv3D(filters=16, kernel_size=(3,3,5), padding='valid', activation='relu')(x)
    x = R2SpectralBlock(x)
    x = R2SpatialBlock(x)
    # flatten 3D to 2D conv input
    shape1 = x.shape[1]; shape2 = x.shape[2]; shape3 = x.shape[3]; shape4 = x.shape[4]
    x = layers.Reshape((shape1, shape2, int(shape3*shape4)))(x)
    x = layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(pool_size=(2,2))(x)
    x = layers.Conv2D(filters=64, kernel_size=(3,3), padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = tf.keras.models.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ---------------- Pipeline ----------------
print("Loading data...")
X_full, y_full = loadData(dataset)
n_classes = int(y_full.max())
print(f"Dataset {dataset}: H={X_full.shape[0]}, W={X_full.shape[1]}, Bands={X_full.shape[2]}, Classes={n_classes}")
total_labeled = int(np.sum(y_full > 0))
print("Total labeled pixels in full map:", total_labeled)

print("Applying PCA ...")
X_pca, pca = applyPCA(X_full, numComponents=K)

# collect coords & labels for ALL labeled pixels (include edges)
coords, labels = [], []
H_img, W_img = X_pca.shape[0], X_pca.shape[1]
for r in range(0, H_img):
    for c in range(0, W_img):
        lab = y_full[r, c]
        if lab > 0:
            coords.append((r, c))          # original coordinates (centers)
            labels.append(lab - 1)         # zero-based labels
coords = np.array(coords, dtype=np.int32)
labels = np.array(labels, dtype=np.int32)
print("Collected coords (should equal total labeled):", len(labels))

# split using exact samples_per_class per class
'''train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet(coords, labels, samples_per_class)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx), "Total:", len(ytrain_idx)+len(ytest_idx))
'''
# split using 10% per class
train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet_ratio(coords, labels, train_ratio)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx))


# create generators
train_gen = PatchGenerator(train_coords, ytrain_idx, X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=True, n_classes=n_classes)
test_gen  = PatchGenerator(test_coords,  ytest_idx,  X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=False, n_classes=n_classes)

# build model
print("Building model...")
model = build_ulite_r2hcn(windowSize, K, n_classes)
model.summary()
params = model.count_params()
print(f"Total parameters: {params:,}")

# save model summary
with open(os.path.join(results_folder, 'ulite_model_summary.txt'), 'w') as f:
    model.summary(print_fn=lambda s: f.write(s + "\n"))

# Training
print("Training...")
tic = time.perf_counter()
history = model.fit(train_gen, validation_data=test_gen, epochs=epochs, verbose=2)
toc = time.perf_counter()
train_time = toc - tic
print(f"Training took {train_time:.2f} s")

# Save training curve
plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.xlabel('epoch'); plt.legend(); plt.title('Training curves')
plt.savefig(os.path.join(results_folder, 'ulite_training_curves.png'), dpi=150)
plt.close()

# Evaluate on test set
print("Evaluating on test set...")
tic1 = time.perf_counter()
y_pred_prob = model.predict(test_gen, verbose=0)
toc1 = time.perf_counter()
test_time = toc1 - tic1
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = ytest_idx

classification = classification_report(y_true, y_pred, digits=4)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
oa = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
each_acc = np.nan_to_num(np.diag(cm) / cm.sum(axis=1))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(y_true, y_pred)

# confusion figure
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='viridis')
plt.colorbar()
plt.title('Confusion Matrix (ULite replica)')
plt.savefig(os.path.join(results_folder, 'ulite_confusion.png'), dpi=150)
plt.close()

# Save results text
with open(os.path.join(results_folder, 'ulite_results.txt'), 'w') as f:
    f.write("ULite-R2HCN (replica) results\n")
    f.write(f"Dataset: {dataset}\n")
    f.write(f"PCA components: {K}, Window size: {windowSize}\n")
    f.write(f"Params: {params}\n")
    f.write(f"Train time (s): {train_time:.2f}\n")
    f.write(f"Test inference time (s total): {test_time:.4f}\n")
    f.write(f"Overall Accuracy (OA): {oa*100:.2f}%\n")
    f.write(f"Average Accuracy (AA): {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n")
    f.write(f"Precision (weighted): {precision*100:.2f}%\n")
    f.write(f"Recall (weighted): {recall*100:.2f}%\n")
    f.write(f"F1-score (weighted): {f1*100:.2f}%\n\n")
    f.write("Classwise accuracies (%):\n")
    f.write(", ".join([f"{x*100:.2f}" for x in each_acc]) + "\n\n")
    f.write("Classification report:\n")
    f.write(classification + "\n")
    f.write("Confusion matrix:\n")
    f.write(np.array2string(cm) + "\n")

# Predict full map (batched) for visualization -- include all labeled pixels
print("Predicting full map (batched)...")
PATCH = windowSize
pad = PATCH // 2
Xp = padWithZeros(X_pca, pad)
H_img, W_img = y_full.shape
outputs = np.zeros((H_img, W_img), dtype=np.int32)  # will store class labels in 1..n_classes, 0 for background

coords_all = [(r, c) for r in range(0, H_img) for c in range(0, W_img) if y_full[r, c] > 0]
B = 2048
for start in range(0, len(coords_all), B):
    batch_coords = coords_all[start:start+B]
    batch = np.empty((len(batch_coords), PATCH, PATCH, K, 1), dtype=np.float32)
    for i, (r, c) in enumerate(batch_coords):
        patch = Xp[r:r+PATCH, c:c+PATCH, :]   # padded Xp: indexed by r..r+PATCH
        batch[i, ..., 0] = patch
    preds = np.argmax(model.predict(batch, verbose=0), axis=1)
    for (r, c), p in zip(batch_coords, preds):
        outputs[r, c] = int(p) + 1  # keep 1-based label for visualization

# Save maps using spectral (background remains 0 -> black)
spectral.save_rgb(os.path.join(results_folder, f'ulite_classified_map_{dataset}.jpg'), outputs.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, f'ulite_ground_truth_{dataset}.jpg'), y_full.astype(int), colors=spectral.spy_colors)

# Save model weights
model.save_weights(os.path.join(results_folder, 'ulite_weights.weights.h5'))

# Zip outputs
zip_path = os.path.join('.', f'ulite_outputs_{dataset}_spc{samples_per_class}.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for fn in os.listdir(results_folder):
        zipf.write(os.path.join(results_folder, fn), arcname=fn)

print("ULite replica pipeline done.")
print("Saved results in folder:", results_folder)
print("Zipped outputs:", zip_path)

Loading data...
Dataset IP: H=145, W=145, Bands=200, Classes=16
Total labeled pixels in full map: 10249
Applying PCA ...
Collected coords (should equal total labeled): 10249
Train samples: 1018 Test samples: 9231
Building model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 25, 25, 30, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d (Conv3D)                 │ (None, 23, 23, 24, 8)  │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 21, 21, 20, 16) │         5,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 21, 21, 20, 16) │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 21, 21, 20, 16) │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_3 (Conv3D)               │ (None, 21, 21, 20, 9)  │           153 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 21, 20, 9)  │            36 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_4 (Conv3D)               │ (None, 21, 21, 20, 16) │         3,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 21, 21, 20, 16) │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 21, 21, 320)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 21, 21, 32)     │        13,152 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 21, 21, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 21, 21, 1, 32)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 21, 21, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 21, 21, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 10, 10, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,005 (222.68 KB)

 Trainable params: 56,859 (222.11 KB)

...

Training...
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


4/4 - 47s - 12s/step - accuracy: 0.0363 - loss: 2.9618 - val_accuracy: 0.0948 - val_loss: 2.7701
Epoch 2/100
4/4 - 43s - 11s/step - accuracy: 0.1316 - loss: 2.6475 - val_accuracy: 0.0948 - val_loss: 2.7654
Epoch 3/100
4/4 - 3s - 751ms/step - accuracy: 0.1916 - loss: 2.5114 - val_accuracy: 0.2394 - val_loss: 2.7608
Epoch 4/100
4/4 - 3s - 743ms/step - accuracy: 0.2112 - loss: 2.4222 - val_accuracy: 0.2394 - val_loss: 2.7554
Epoch 5/100
4/4 - 3s - 747ms/step - accuracy: 0.2505 - loss: 2.2794 - val_accuracy: 0.2394 - val_loss: 2.7499
Epoch 6/100
4/4 - 3s - 826ms/step - accuracy: 0.2917 - loss: 2.1752 - val_accuracy: 0.2394 - val_loss: 2.7449
Epoch 7/100
4/4 - 5s - 1s/step - accuracy: 0.3369 - loss: 2.0355 - val_accuracy: 0.3141 - val_loss: 2.7377
Epoch 8/100
4/4 - 3s - 763ms/step - accuracy: 0.3664 - loss: 1.9255 - val_accuracy: 0.3330 - val_loss: 2.7290
Epoch 9/100
4/4 - 3s - 772ms/step - accuracy: 0.4204 - loss: 1.7465 - val_accuracy: 0.4039 - val_loss: 2.7172
Epoch 10/100
4/4 - 3s - 789

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Predicting full map (batched)...
ULite replica pipeline done.
Saved results in folder: ulite_results_IP_spc15
Zipped outputs: ./ulite_outputs_IP_spc15.zip
